# pulling metadata information from maps layer and its sublayers.


In [1]:
from arcgis.gis import GIS
import pandas as pd
from dotenv import load_dotenv
import os
import json

load_dotenv()


signin_name = os.getenv('ESRI_USERNAME')
signin_password = os.getenv('ESRI_PASSWORD')
signin_link = os.getenv("HUBLINK")

# Connect to the organization
gis = GIS(signin_link, signin_name, signin_password )
username = gis.users.me.username

print("Done")


Done


In [2]:
maps = {
	'natural-treasures': {
		'title': 'Natural Treasures',
		'mapId': 'aba702c5420e4a36ac645f14a00ba8f1'
	},
	'transportation-infrastructure': {
		'title': 'Transportation + Infrastructure',
		'mapId': '25c75ac490b94b85b02aa4fd2f341fbb'
	}
}

## Find layer metadata - nested

In [28]:
skip_titles = {'Cross Sector Data', 'Thrive County Boundaries',
               'Thrive watershed region', 'waterbodies'}

rows = []

for slug, m in maps.items():
    map_item = gis.content.get(m['mapId'])
    wm = map_item.get_data()
    for top in wm.get('operationalLayers', []):
        top_title = str(top.get('title'))
        if top_title in skip_titles:
            continue
        is_group = top.get('layerType') == 'GroupLayer'
        if is_group and not top.get('layers'):
            continue
        group = top_title if is_group else ''
        for lyr in (top.get('layers') or []) if is_group else [top]:
            layer_title = str(lyr.get('title'))
            if layer_title in skip_titles:
                continue
            url = lyr.get('url') or ''
            # one row per map entry, plus one row for the service layer that entry draws.
            # the entry url ends in that layer's index, so no item.layers call is needed here
            # and nothing gets repeated under the wrong parent. image services have no index.
            tail = url.rstrip('/').split('/')[-1]
            base = {
                'map_slug': slug,
                'map_title': m['title'],
                'map_id': m['mapId'],
                'group': group,
                'layer_title': layer_title,
                'layer_type': lyr.get('layerType'),
                'item_id': lyr.get('itemId'),
                'url': url,
            }
            rows.append({**base, 'level': 'layer',
                         'sublayer_id': None, 'sublayer_name': None})
            if tail.isdigit():
                rows.append({**base, 'level': 'sublayer',
                             'sublayer_id': int(tail), 'sublayer_name': None})
    break

df = pd.DataFrame(rows)
print(len(df), 'rows |', (df.level == 'layer').sum(), 'layer |', (df.level == 'sublayer').sum(), 'sublayer |',
      df.item_id.nunique(), 'unique items |',
      df[['item_id', 'sublayer_id']].dropna().drop_duplicates().shape[0], 'unique item+sublayer pairs')
df


38 rows | 21 layer | 17 sublayer | 19 unique items | 15 unique item+sublayer pairs


,map_slug,map_title,map_id,group,layer_title,layer_type,item_id,url,level,sublayer_id,sublayer_name
0,natural-treasures,Natural Treasures,aba702c5420e4a36ac645f14a00ba8f1,Land Use Change,"Developed Land Change 2004–2024, Percent Per Mile",ArcGISFeatureLayer,c8cb5376347542f597437060984d4069,https://services3.arcgis.com/xpR2E2r2KmCE5hF3/arcgis/res...,layer,NaN,None
1,natural-treasures,Natural Treasures,aba702c5420e4a36ac645f14a00ba8f1,Land Use Change,"Developed Land Change 2004–2024, Percent Per Mile",ArcGISFeatureLayer,c8cb5376347542f597437060984d4069,https://services3.arcgis.com/xpR2E2r2KmCE5hF3/arcgis/res...,sublayer,0.0,None
2,natural-treasures,Natural Treasures,aba702c5420e4a36ac645f14a00ba8f1,Land Use Change,Natural Cover Change 2004–2024 percent mile,ArcGISFeatureLayer,c8cb5376347542f597437060984d4069,https://services3.arcgis.com/xpR2E2r2KmCE5hF3/arcgis/res...,layer,NaN,None
3,natural-treasures,Natural Treasures,aba702c5420e4a36ac645f14a00ba8f1,Land Use Change,Natural Cover Change 2004–2024 percent mile,ArcGISFeatureLayer,c8cb5376347542f597437060984d4069,https://services3.arcgis.com/xpR2E2r2KmCE5hF3/arcgis/res...,sublayer,0.0,None
4,natural-treasures,Natural Treasures,aba702c5420e4a36ac645f14a00ba8f1,Land Use Change,Land Use Change 2004 – 2024 60 meter resolution,ArcGISTiledImageServiceLayer,d54eeaddbc3e442eae779aacc06fdf7c,https://tiledimageservices3.arcgis.com/xpR2E2r2KmCE5hF3/...,layer,NaN,None
5,natural-treasures,Natural Treasures,aba702c5420e4a36ac645f14a00ba8f1,Land Use Change,Future Land Cover: 2050,ArcGISImageServiceLayer,13c415b776c0463e863f89361ed688da,https://env1.arcgis.com/arcgis/rest/services/Land_Cover_...,layer,NaN,None
6,natural-treasures,Natural Treasures,aba702c5420e4a36ac645f14a00ba8f1,Land Use Change,thrive_watershed_huc10_boundary,ArcGISFeatureLayer,49f5a7a36df7476fb13a1daec41f8714,https://services3.arcgis.com/xpR2E2r2KmCE5hF3/arcgis/res...,layer,NaN,None
7,natural-treasures,Natural Treasures,aba702c5420e4a36ac645f14a00ba8f1,Land Use Change,thrive_watershed_huc10_boundary,ArcGISFeatureLayer,49f5a7a36df7476fb13a1daec41f8714,https://services3.arcgis.com/xpR2E2r2KmCE5hF3/arcgis/res...,sublayer,0.0,None
8,natural-treasures,Natural Treasures,aba702c5420e4a36ac645f14a00ba8f1,Agriculture & Working Lands,"Agricultural Land Change, Percent Per Mile",ArcGISFeatureLayer,c8cb5376347542f597437060984d4069,https://services3.arcgis.com/xpR2E2r2KmCE5hF3/arcgis/res...,layer,NaN,None
9,natural-treasures,Natural Treasures,aba702c5420e4a36ac645f14a00ba8f1,Agriculture & Working Lands,"Agricultural Land Change, Percent Per Mile",ArcGISFeatureLayer,c8cb5376347542f597437060984d4069,https://services3.arcgis.com/xpR2E2r2KmCE5hF3/arcgis/res...,sublayer,0.0,None


In [29]:
# test every item in the list: its text, and the rows that carry it
for iid in df['item_id'].dropna().unique():
    item = gis.content.get(iid)
    print('=' * 88)
    if item is None:
        print(iid, '-> the item did not come back')
        continue
    print(iid, '|', item.get('title'), '|', item.get('owner'), '|', item.get('access'))
    print('  summary        :', repr(item.get('snippet'))[:70])
    print('  description    :', repr(item.get('description'))[:70])
    print('  acknowledgments:', repr(item.get('accessInformation'))[:70])
    for r in df[df.item_id == iid].itertuples():
        if pd.isna(r.sublayer_id):
            link = f"{gis.url}/home/item.html?id={iid}"
        else:
            link = f"{gis.url}/home/item.html?id={iid}&sublayer={int(r.sublayer_id)}"
        print(f'    {r.level:8s} | {str(r.layer_title)[:42]:42s} | {link}')


c8cb5376347542f597437060984d4069 | hex_grid_1mi_landcover_change_watershed | sara2263_thrive_geohub | public
  summary        : None
  description    : None
  acknowledgments: None
    layer    | Developed Land Change 2004–2024, Percent P | https://thrive-geohub.maps.arcgis.com/home/item.html?id=c8cb5376347542f597437060984d4069
    sublayer | Developed Land Change 2004–2024, Percent P | https://thrive-geohub.maps.arcgis.com/home/item.html?id=c8cb5376347542f597437060984d4069&sublayer=0
    layer    | Natural Cover Change 2004–2024 percent mil | https://thrive-geohub.maps.arcgis.com/home/item.html?id=c8cb5376347542f597437060984d4069
    sublayer | Natural Cover Change 2004–2024 percent mil | https://thrive-geohub.maps.arcgis.com/home/item.html?id=c8cb5376347542f597437060984d4069&sublayer=0
    layer    | Agricultural Land Change, Percent Per Mile | https://thrive-geohub.maps.arcgis.com/home/item.html?id=c8cb5376347542f597437060984d4069
    sublayer | Agricultural Land Change, Percent Per

In [30]:
# needs df from the cell above.
# Only what a person edits in the portal. Column names are the portal's own labels.
# item page fields: Title, Summary, Description, Acknowledgments, Sharing
# a layer row and a sublayer row both exist, so the reviewer knows there are two places to look.
item_rows = []

for iid in df['item_id'].dropna().unique():
    item = gis.content.get(iid)
    if item is None:
        continue
    item_rows.append({
        'item_id':         iid,
        'item_title':      item.get('title'),
        'summary':         item.get('snippet'),          # portal label: Summary
        'description':     item.get('description'),      # portal label: Description
        'acknowledgments': item.get('accessInformation'),  # portal label: Acknowledgments
        'owner':           item.get('owner'),
        'sharing':         item.get('access'),           # portal label: Sharing
    })

items = pd.DataFrame(item_rows)
layers = df.drop(columns=['sublayer_name']).merge(items, on='item_id', how='left')

# the page to open in order to edit this row
layers['edit_link'] = gis.url + '/home/item.html?id=' + layers['item_id'].fillna('')
sub_layer = layers['sublayer_id'].notna()
layers.loc[sub_layer, 'edit_link'] = (layers.loc[sub_layer, 'edit_link'] + '&sublayer='
                                      + layers.loc[sub_layer, 'sublayer_id'].astype(int).astype(str))

# which editable fields are still empty on this row
fields = ['summary', 'description', 'acknowledgments']
layers['missing'] = layers[fields].fillna('').apply(
    lambda r: ', '.join(f for f in fields if not str(r[f]).strip()), axis=1)

# what to be careful about on this row
layers['notes'] = ''
layers.loc[sub_layer, 'notes'] = ("values shown are the item text. A sublayer's own summary and "
                                  "description are not readable through the API, check them on this page.")

# the REST url is developer-only, so it is not part of the deliverable
keep = ['map_slug', 'group', 'layer_title', 'level', 'sublayer_id', 'item_id', 'item_title',
        'summary', 'description', 'acknowledgments', 'owner', 'sharing', 'missing', 'notes', 'edit_link']
layers = layers[keep]

print(layers.shape)
print('rows with nothing missing  :', layers.missing.eq('').sum(), 'of', len(layers))
print('rows missing summary       :', layers.missing.str.contains('summary').sum())
print('rows missing description   :', layers.missing.str.contains('description').sum())
print('rows missing acknowledgments:', layers.missing.str.contains('acknowledgments').sum())
print('rows editable by us (owner is the Thrive account):', layers.owner.eq('sara2263_thrive_geohub').sum())
layers


(38, 15)
rows with nothing missing  : 5 of 38
rows missing summary       : 13
rows missing description   : 22
rows missing acknowledgments: 33
rows editable by us (owner is the Thrive account): 24


,map_slug,group,layer_title,level,sublayer_id,item_id,item_title,summary,description,acknowledgments,owner,sharing,missing,notes,edit_link
0,natural-treasures,Land Use Change,"Developed Land Change 2004–2024, Percent Per Mile",layer,NaN,c8cb5376347542f597437060984d4069,hex_grid_1mi_landcover_change_watershed,NaN,NaN,NaN,sara2263_thrive_geohub,public,"summary, description, acknowledgments",,https://thrive-geohub.maps.arcgis.com/home/item.html?id=...
1,natural-treasures,Land Use Change,"Developed Land Change 2004–2024, Percent Per Mile",sublayer,0.0,c8cb5376347542f597437060984d4069,hex_grid_1mi_landcover_change_watershed,NaN,NaN,NaN,sara2263_thrive_geohub,public,"summary, description, acknowledgments",values shown are the item text. A sublayer's own summary...,https://thrive-geohub.maps.arcgis.com/home/item.html?id=...
2,natural-treasures,Land Use Change,Natural Cover Change 2004–2024 percent mile,layer,NaN,c8cb5376347542f597437060984d4069,hex_grid_1mi_landcover_change_watershed,NaN,NaN,NaN,sara2263_thrive_geohub,public,"summary, description, acknowledgments",,https://thrive-geohub.maps.arcgis.com/home/item.html?id=...
3,natural-treasures,Land Use Change,Natural Cover Change 2004–2024 percent mile,sublayer,0.0,c8cb5376347542f597437060984d4069,hex_grid_1mi_landcover_change_watershed,NaN,NaN,NaN,sara2263_thrive_geohub,public,"summary, description, acknowledgments",values shown are the item text. A sublayer's own summary...,https://thrive-geohub.maps.arcgis.com/home/item.html?id=...
4,natural-treasures,Land Use Change,Land Use Change 2004 – 2024 60 meter resolution,layer,NaN,d54eeaddbc3e442eae779aacc06fdf7c,landuse_change_watershed_boundary_2004_2024_60m,,,,sara2263_thrive_geohub,public,"summary, description, acknowledgments",,https://thrive-geohub.maps.arcgis.com/home/item.html?id=...
5,natural-treasures,Land Use Change,Future Land Cover: 2050,layer,NaN,13c415b776c0463e863f89361ed688da,Land Cover 2050 - Country,"Predicted land cover for each country for year 2050, gen...",NaN,NaN,sara2263_thrive_geohub,public,"description, acknowledgments",,https://thrive-geohub.maps.arcgis.com/home/item.html?id=...
6,natural-treasures,Land Use Change,thrive_watershed_huc10_boundary,layer,NaN,49f5a7a36df7476fb13a1daec41f8714,thrive_watershed_huc10_boundary,NaN,NaN,NaN,sara2263_thrive_geohub,public,"summary, description, acknowledgments",,https://thrive-geohub.maps.arcgis.com/home/item.html?id=...
7,natural-treasures,Land Use Change,thrive_watershed_huc10_boundary,sublayer,0.0,49f5a7a36df7476fb13a1daec41f8714,thrive_watershed_huc10_boundary,NaN,NaN,NaN,sara2263_thrive_geohub,public,"summary, description, acknowledgments",values shown are the item text. A sublayer's own summary...,https://thrive-geohub.maps.arcgis.com/home/item.html?id=...
8,natural-treasures,Agriculture & Working Lands,"Agricultural Land Change, Percent Per Mile",layer,NaN,c8cb5376347542f597437060984d4069,hex_grid_1mi_landcover_change_watershed,NaN,NaN,NaN,sara2263_thrive_geohub,public,"summary, description, acknowledgments",,https://thrive-geohub.maps.arcgis.com/home/item.html?id=...
9,natural-treasures,Agriculture & Working Lands,"Agricultural Land Change, Percent Per Mile",sublayer,0.0,c8cb5376347542f597437060984d4069,hex_grid_1mi_landcover_change_watershed,NaN,NaN,NaN,sara2263_thrive_geohub,public,"summary, description, acknowledgments",values shown are the item text. A sublayer's own summary...,https://thrive-geohub.maps.arcgis.com/home/item.html?id=...


In [31]:
layers.to_excel( "../PY/layers_metadata.xlsx" )

## Accessing group layers

In [7]:
nta_map = maps['natural-treasures']

In [22]:
skip_titles = {'Cross Sector Data', 'Thrive County Boundaries',
               'Thrive watershed region', 'waterbodies'}

rows = []

map_item = gis.content.get( nta_map['mapId'])
wm = map_item.get_data()

wm

{'operationalLayers': [{'id': '19f8049271e-layer-46',
   'title': 'Cross Sector Data',
   'layers': [{'id': '19dd3d17b2b-layer-97',
     'title': 'parcels',
     'visibility': True,
     'layers': [{'id': '19dd1a4472c-layer-42',
       'title': 'Tennessee Property Boundaries Public Use',
       'url': 'https://services1.arcgis.com/YuVBSS7Y1of2Qud1/arcgis/rest/services/Tennessee_Property_Boundaries_Public_Use/FeatureServer/0',
       'visibility': False,
       'itemId': 'c4a29816b37f4824b71dad7b3c7a1b90',
       'layerType': 'ArcGISFeatureLayer',
       'showLabels': False,
       'layerDefinition': {'drawingInfo': {'renderer': {'type': 'simple',
          'visualVariables': [{'type': 'sizeInfo',
            'valueExpression': '$view.scale',
            'stops': [{'size': 0.21485714285714283, 'value': 2929},
             {'size': 0.10742857142857141, 'value': 9152},
             {'size': 0.05371428571428571, 'value': 36606},
             {'size': 0, 'value': 73213}],
            'targe

In [23]:
for i in wm['operationalLayers']:

    print(f"""
    --------------------------
    {i['title']}, 
    {i['id']}, 
    {i['itemId'] if 'itemId' in i.keys() else 'No item id'}
    {i['layerType']}
    -------------------------
    """)


    --------------------------
    Cross Sector Data, 
    19f8049271e-layer-46, 
    No item id
    GroupLayer
    -------------------------
    

    --------------------------
    Land Use Change, 
    19dd37b8622-layer-91, 
    No item id
    GroupLayer
    -------------------------
    

    --------------------------
    Agriculture & Working Lands, 
    19dca4c8b81-layer-18, 
    No item id
    GroupLayer
    -------------------------
    

    --------------------------
    Biodiversity & Habitat, 
    19f63a6445b-layer-39, 
    No item id
    GroupLayer
    -------------------------
    

    --------------------------
    Protected Lands, 
    19dca4b8b42-layer-16, 
    d5ee129f6f9f477194721b2cda6f8049
    GroupLayer
    -------------------------
    

    --------------------------
    waterbodies, 
    1a06dc0088f-layer-38, 
    3ca7f70273f74d2a89982e5cecee4d98
    ArcGISFeatureLayer
    -------------------------
    

    --------------------------
    Water Resources, 
 